# 11.29 — Offline RL

Offline reinforcement learning learns a better policy from a fixed log of past behavior, without collecting new trials. That makes the math look like ordinary value learning, but the central danger is different: if the learned policy chooses actions the dataset barely covered, bootstrapped values can extrapolate confidently into places where there is no evidence.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build offline RL one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is shown with tiny NumPy tables. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, tabular value functions, probabilities, and small simulations.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for stochastic demos.

### 1. A fixed dataset is the whole world the learner gets

Offline RL starts with logged tuples `(state, action, reward, next_state)`. Unlike online RL, the learner cannot try a missing action to see what happens. The behavior policy that produced the data therefore defines **support**: actions with many samples are grounded, and actions with zero samples are guesses no matter how attractive their current value estimate looks.

In [ ]:
states_w = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # logged states from a fixed behavior policy.
actions_w = np.array([0, 0, 0, 1, 0, 1, 1, 1])  # action 1 is rare in state 0; action 0 is rare in state 1.
rewards_w = np.array([1.0, 1.2, 0.8, 1.5, 0.2, 2.0, 1.8, 2.2])  # observed one-step rewards.
next_states_w = np.array([0, 0, 1, 1, 0, 1, 1, 1])  # observed next states.
counts_w = np.zeros((2, 2), dtype=int)  # rows are states, columns are actions.
for s_w, a_w in zip(states_w, actions_w):
    counts_w[s_w, a_w] += 1  # count behavior-policy support for each state-action pair.

print("support counts:\n", counts_w)

assert counts_w.tolist() == [[3, 1], [1, 3]]

▶ What you'll see: the diagonal-ish behavior coverage tells us which actions the dataset really knows about.

In [ ]:
behavior_w = counts_w / counts_w.sum(axis=1, keepdims=True)  # empirical behavior policy μ(a|s).

print("behavior policy μ(a|s):\n", np.round(behavior_w, 3))

plt.figure(figsize=(4.5, 3))
plt.imshow(behavior_w, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="μ(a|s)")
plt.xticks([0, 1], ["a0", "a1"]); plt.yticks([0, 1], ["s0", "s1"])
plt.title("1: dataset support from the behavior policy"); plt.show()

▶ What you'll see: state 0 mostly used action 0, while state 1 mostly used action 1.

*Why it's done this way:* Offline RL is a support-limited problem. The empirical behavior policy is not merely descriptive; it is the map of where estimates are trustworthy. Later conservative and behavior-regularized updates deliberately use this map to stop the learned policy from exploiting unsupported action values.

### 2. Return and Bellman targets value delayed consequence

RL is not just immediate reward. A return adds discounted future rewards, and a Bellman target replaces the unknown full future with a current estimate of the next state's value. In offline RL this target is useful but dangerous: it reuses the learner's own estimates, so an unsupported overestimate can be copied backward through the table.

In [ ]:
rewards_path_w = np.array([1.0, 0.0, 2.0])  # a tiny reward sequence.
gamma_w = 0.9  # future rewards matter but are discounted.
discounts_w = gamma_w ** np.arange(len(rewards_path_w))  # [1, γ, γ²].
return_w = float(np.sum(discounts_w * rewards_path_w))  # discounted return G.

print("discounts:", np.round(discounts_w, 3))
print("three-step return:", round(return_w, 3))

assert round(return_w, 3) == 2.620

▶ What you'll see: the delayed reward two steps away counts as `0.9²·2 = 1.62`.

In [ ]:
r_w = 1.0  # observed immediate reward.
next_value_w = 0.8  # current estimate V(s').
q_old_w = 0.4  # old action-value estimate.
alpha_w = 0.5  # partial update size.
target_w = r_w + gamma_w * next_value_w  # one-step Bellman target.
q_new_w = q_old_w + alpha_w * (target_w - q_old_w)  # move partway toward the target.

print("target:", round(target_w, 3), "updated Q:", round(q_new_w, 3))

assert round(target_w, 3) == 1.720 and round(q_new_w, 3) == 1.060

▶ What you'll see: the update moves halfway from 0.4 toward 1.72.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["old Q", "target", "new Q"], [q_old_w, target_w, q_new_w], color=["gray", "black", "seagreen"])
plt.ylabel("value"); plt.title("2: Bellman target and partial update"); plt.show()

▶ What you'll see: the new estimate is between the old estimate and the bootstrap target, not a full overwrite.

*Why it's done this way:* The Bellman equation decomposes consequence into `now + discounted later`, which reduces variance compared with waiting for complete returns. The price is bootstrap bias: because the target contains a learned estimate, offline RL must control which next-action values are allowed to influence it.

### 3. Distribution shift appears when the learned policy leaves the log

A policy turns action values into probabilities. If the learned policy places probability mass on actions that the behavior policy rarely sampled, the learner is asking the dataset to answer counterfactual questions it did not record. That mismatch is distribution shift: training data came from μ, but decisions are evaluated under π.

In [ ]:
logits_w = np.array([1.0, 0.0])  # unnormalized policy preferences.
exp_w = np.exp(logits_w - np.max(logits_w))  # stable softmax numerator.
policy_w = exp_w / exp_w.sum()  # π(a|s).
expected_reward_w = float(policy_w @ np.array([2.0, 0.0]))  # expected one-step reward under π.

print("softmax policy:", np.round(policy_w, 3))
print("expected reward:", round(expected_reward_w, 3))

assert np.allclose(np.round(policy_w, 3), [0.731, 0.269])
assert round(expected_reward_w, 3) == 1.462

▶ What you'll see: action 0 gets about 73.1% probability, so reward 2 contributes about 1.462 in expectation.

In [ ]:
pi_shift_w = np.array([[0.10, 0.90], [0.20, 0.80]])  # a learned policy that chases action 1.
ratio_w = pi_shift_w / np.maximum(behavior_w, 1e-9)  # importance ratios π/μ reveal shift.

print("π/μ ratios:\n", np.round(ratio_w, 2))

assert round(float(ratio_w[0, 1]), 2) == 3.60

▶ What you'll see: in state 0, the learned policy uses action 1 about 3.6× more than the data did.

In [ ]:
plt.figure(figsize=(5, 3))
x_w = np.arange(2)
plt.bar(x_w - 0.18, behavior_w[0], width=0.36, label="behavior μ", color="gray")
plt.bar(x_w + 0.18, pi_shift_w[0], width=0.36, label="learned π", color="crimson")
plt.xticks(x_w, ["a0", "a1"]); plt.ylabel("probability"); plt.title("3: distribution shift in state 0")
plt.legend(); plt.show()

▶ What you'll see: π puts most mass on the action that μ barely sampled in state 0.

*Why it's done this way:* Ratios make distribution shift numeric. A ratio near 1 means evaluation resembles the logged data; a large ratio means a small, noisy part of the dataset is being amplified. Offline algorithms regularize policies or values precisely to keep those ratios from exploding.

### 4. Conservative Q values penalize unsupported actions

A standard Bellman backup can overestimate actions with little data because max over actions selects the largest estimate, including estimates that are large by noise. Conservative Q-learning counters that by pushing down all action values, then giving observed actions a compensating lift. In tabular form, this acts like a penalty on actions outside the data support.

In [ ]:
Q_w = np.array([[1.0, 3.0], [0.5, 2.0]])  # current action values; s0,a1 looks tempting.
mean_logged_reward_w = np.zeros((2, 2))  # empirical rewards where observed.
for s_w in range(2):
    for a_w in range(2):
        mask_w = (states_w == s_w) & (actions_w == a_w)
        mean_logged_reward_w[s_w, a_w] = rewards_w[mask_w].mean() if np.any(mask_w) else 0.0

print("current Q:\n", Q_w)
print("logged reward means:\n", np.round(mean_logged_reward_w, 3))

▶ What you'll see: the tempting high value for `(s0,a1)` is based on only one logged transition.

In [ ]:
coverage_w = counts_w / np.maximum(counts_w.sum(axis=1, keepdims=True), 1)  # per-state action frequencies.
penalty_w = 0.8 * (1.0 - coverage_w)  # bigger penalty where behavior support is weaker.
Q_conservative_w = Q_w - penalty_w  # one simple conservative adjustment.

print("support-aware penalty:\n", np.round(penalty_w, 3))
print("conservative Q:\n", np.round(Q_conservative_w, 3))

assert round(float(Q_conservative_w[0, 1]), 3) == 2.400

▶ What you'll see: the low-support action in state 0 is pushed down more than the well-supported action.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["raw Q(s0,a0)", "raw Q(s0,a1)", "cons a0", "cons a1"],
        [Q_w[0, 0], Q_w[0, 1], Q_conservative_w[0, 0], Q_conservative_w[0, 1]],
        color=["gray", "gray", "seagreen", "crimson"])
plt.xticks(rotation=20); plt.ylabel("value"); plt.title("4: conservatism lowers weak-support actions"); plt.show()

▶ What you'll see: action 1 still may be good, but its unsupported optimism is reduced.

*Why it's done this way:* The mathematical reason for conservatism is the max operator's selection bias: among noisy estimates, the largest is often overestimated. Penalizing low-support actions changes the fixed point so the learner prefers values it can justify from the dataset rather than values created by extrapolation error.

### 5. Behavior regularization keeps policy improvement near the data

A policy can improve without becoming arbitrary. Behavior regularization adds a cost for moving too far from the behavior policy, commonly with a KL-style penalty. The objective is no longer just expected Q; it is expected Q minus a distance-from-data term.

In [ ]:
q_s0_w = Q_conservative_w[0]  # conservative action values in state 0.
mu_s0_w = behavior_w[0]  # behavior probabilities in state 0.
candidate_policies_w = np.array([[0.75, 0.25], [0.50, 0.50], [0.20, 0.80]])  # increasingly shifted policies.
values_w = candidate_policies_w @ q_s0_w  # expected conservative Q for each candidate.
kl_w = np.sum(candidate_policies_w * np.log(np.maximum(candidate_policies_w, 1e-9) / mu_s0_w), axis=1)  # KL(π||μ).

print("expected Q:", np.round(values_w, 3))
print("KL from behavior:", np.round(kl_w, 3))

▶ What you'll see: the most shifted policy gets high value but also pays the largest KL cost.

In [ ]:
beta_w = 1.2  # regularization strength.
objective_w = values_w - beta_w * kl_w  # behavior-regularized improvement objective.
best_idx_w = int(np.argmax(objective_w))

print("regularized objective:", np.round(objective_w, 3))
print("chosen policy:", candidate_policies_w[best_idx_w])

assert best_idx_w == 1

▶ What you'll see: the middle policy wins because it improves value without moving too far off-support.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(["near μ", "middle", "shifted"], values_w, marker="o", label="expected Q")
plt.plot(["near μ", "middle", "shifted"], objective_w, marker="s", label="Q - β KL")
plt.ylabel("score"); plt.title("5: behavior regularization changes the winner")
plt.legend(); plt.show()

▶ What you'll see: raw value prefers shifting hard, while the regularized score prefers a safer middle policy.

*Why it's done this way:* The KL term is a Lagrange-style way to encode the offline constraint `C(π) ≤ d`. It does not say the behavior policy is optimal; it says deviations must pay rent because every extra unit of shift asks the fixed dataset to support more counterfactual reasoning.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, each tiny offline RL toy isolates one computational mechanic with small numbers, printed intermediates, one picture, and an `assert` that pins the result.

### ✍️ Toy 1 · Logged support defines the behavior policy

In offline RL, the fixed log tells us which state-action pairs are supported and how often the behavior policy chose them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_states = np.array([0, 0, 0, 1, 1, 1])                   # -> [0, 0, 0, 1, 1, 1]
t1_actions = np.array([0, 0, 1, 0, 1, 1])                  # -> [0, 0, 1, 0, 1, 1]
t1_counts = np.zeros((2, 2), dtype=int)
for t1_s, t1_a in zip(t1_states, t1_actions):
    t1_counts[t1_s, t1_a] += 1
t1_behavior = t1_counts / t1_counts.sum(axis=1, keepdims=True) # -> [[0.667, 0.333], [0.333, 0.667]]

print("support counts:", t1_counts.tolist())              # -> [[2, 1], [1, 2]]
print("behavior policy:", np.round(t1_behavior, 3).tolist()) # -> [[0.667, 0.333], [0.333, 0.667]]

assert t1_counts.tolist() == [[2, 1], [1, 2]]

plt.figure(figsize=(4.2, 2.8))
plt.imshow(t1_behavior, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="μ(a|s)")
plt.xticks([0, 1], ["a0", "a1"])
plt.yticks([0, 1], ["s0", "s1"])
plt.title("Toy 1 · behavior support")
plt.show()

▶ What you'll see: state `0` mostly supports action `0`, while state `1` mostly supports action `1`.

### ✍️ Toy 2 · Discounted return adds delayed rewards

A return multiplies each future reward by a discount power before summing.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_rewards = np.array([2.0, 0.0, 1.0])                     # -> [2.0, 0.0, 1.0]
t2_gamma = 0.5                                             # -> 0.5
t2_discounts = t2_gamma ** np.arange(t2_rewards.size)      # -> [1.0, 0.5, 0.25]
t2_weighted = t2_discounts * t2_rewards                    # -> [2.0, 0.0, 0.25]
t2_return = float(t2_weighted.sum())                       # -> 2.25

print("discounts:", t2_discounts.tolist())                # -> [1.0, 0.5, 0.25]
print("weighted rewards:", t2_weighted.tolist())          # -> [2.0, 0.0, 0.25]
print("return:", t2_return)                               # -> 2.25

assert t2_return == 2.25

plt.figure(figsize=(4.4, 2.8))
plt.bar(range(3), t2_weighted, color="teal")
plt.xlabel("time offset")
plt.ylabel("discounted reward")
plt.title("Toy 2 · delayed reward is discounted")
plt.show()

▶ What you'll see: the last reward contributes only `0.25` because it is two discount steps away.

### ✍️ Toy 3 · Bellman update blends old Q with a target

A one-step target is not a full overwrite; the learning rate moves the old estimate partway toward it.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_reward = 1.0                                            # -> 1.0
t3_next_value = 2.0                                        # -> 2.0
t3_gamma = 0.5                                             # -> 0.5
t3_old_q = 0.5                                             # -> 0.5
t3_alpha = 0.25                                            # -> 0.25
t3_target = t3_reward + t3_gamma * t3_next_value           # -> 2.0
t3_td_error = t3_target - t3_old_q                         # -> 1.5
t3_new_q = t3_old_q + t3_alpha * t3_td_error               # -> 0.875

print("target:", t3_target)                               # -> 2.0
print("TD error:", t3_td_error)                           # -> 1.5
print("new Q:", t3_new_q)                                 # -> 0.875

assert t3_new_q == 0.875

plt.figure(figsize=(4.4, 2.8))
plt.bar(["old Q", "target", "new Q"], [t3_old_q, t3_target, t3_new_q], color=["gray", "black", "seagreen"])
plt.ylabel("value")
plt.title("Toy 3 · partial Bellman update")
plt.show()

▶ What you'll see: `new Q = 0.875`, between the old estimate `0.5` and the target `2.0`.

### ✍️ Toy 4 · Policy shift ratios flag weak support

A learned policy can put much more mass on an action than the behavior policy did; the ratio `π/μ` makes that visible.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_behavior = np.array([0.75, 0.25])                       # -> [0.75, 0.25]
t4_logits = np.array([-1.0, 1.0])                          # -> [-1.0, 1.0]
t4_exp = np.exp(t4_logits - np.max(t4_logits))             # -> [0.135, 1.0]
t4_policy = t4_exp / t4_exp.sum()                          # -> [0.119, 0.881]
t4_ratio = t4_policy / t4_behavior                         # -> [0.159, 3.523]

print("behavior μ:", t4_behavior.tolist())                # -> [0.75, 0.25]
print("learned π:", np.round(t4_policy, 3).tolist())      # -> [0.119, 0.881]
print("π/μ ratio:", np.round(t4_ratio, 3).tolist())       # -> [0.159, 3.523]

assert t4_ratio[1] > 3.0

plt.figure(figsize=(4.4, 2.8))
plt.bar(np.arange(2) - 0.18, t4_behavior, width=0.36, label="behavior", color="gray")
plt.bar(np.arange(2) + 0.18, t4_policy, width=0.36, label="learned", color="crimson")
plt.xticks([0, 1], ["a0", "a1"])
plt.ylabel("probability")
plt.title("Toy 4 · learned policy leaves the log")
plt.legend()
plt.show()

▶ What you'll see: action `1` is used more than `3×` as often by the learned policy as by the log.

### ✍️ Toy 5 · Conservative penalty lowers weak-support Q

A support-aware penalty subtracts more from actions that appeared less often in the dataset.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_coverage = np.array([0.75, 0.25])                       # -> [0.75, 0.25]
t5_raw_q = np.array([1.0, 3.0])                            # -> [1.0, 3.0]
t5_strength = 0.8                                          # -> 0.8
t5_penalty = t5_strength * (1.0 - t5_coverage)             # -> [0.2, 0.6]
t5_conservative_q = t5_raw_q - t5_penalty                  # -> [0.8, 2.4]

print("coverage:", t5_coverage.tolist())                  # -> [0.75, 0.25]
print("penalty:", np.round(t5_penalty, 3).tolist())       # -> [0.2, 0.6]
print("conservative Q:", np.round(t5_conservative_q, 3).tolist()) # -> [0.8, 2.4]

assert np.allclose(t5_conservative_q, [0.8, 2.4])

plt.figure(figsize=(4.6, 2.8))
plt.bar(["raw a0", "raw a1", "cons a0", "cons a1"], [t5_raw_q[0], t5_raw_q[1], t5_conservative_q[0], t5_conservative_q[1]], color=["gray", "gray", "seagreen", "crimson"])
plt.xticks(rotation=20)
plt.ylabel("Q value")
plt.title("Toy 5 · weaker support pays a larger penalty")
plt.show()

▶ What you'll see: action `1` still has high value, but it is pushed down more because its coverage is lower.

### ✍️ Toy 6 · Behavior regularization can change the winner

A candidate policy with the highest raw Q can lose once it pays a KL penalty for moving too far from the behavior policy.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_q = np.array([1.0, 2.0])                                # -> [1.0, 2.0]
t6_behavior = np.array([0.75, 0.25])                       # -> [0.75, 0.25]
t6_candidates = np.array([[0.75, 0.25], [0.50, 0.50], [0.20, 0.80]]) # -> near, middle, shifted
t6_values = t6_candidates @ t6_q                           # -> [1.25, 1.5, 1.8]
t6_kl = np.sum(t6_candidates * np.log(t6_candidates / t6_behavior), axis=1) # -> [0.0, 0.144, 0.666]
t6_beta = 1.4                                              # -> 1.4
t6_objective = t6_values - t6_beta * t6_kl                 # -> [1.25, 1.299, 0.867]
t6_best = int(np.argmax(t6_objective))                     # -> 1

print("raw values:", np.round(t6_values, 3).tolist())     # -> [1.25, 1.5, 1.8]
print("KL penalties:", np.round(t6_kl, 3).tolist())       # -> [0.0, 0.144, 0.666]
print("regularized objective:", np.round(t6_objective, 3).tolist()) # -> [1.25, 1.299, 0.867]
print("best candidate index:", t6_best)                   # -> 1

assert t6_best == 1

plt.figure(figsize=(4.8, 2.8))
plt.plot(["near", "middle", "shifted"], t6_values, marker="o", label="raw Q")
plt.plot(["near", "middle", "shifted"], t6_objective, marker="s", label="Q - β KL")
plt.ylabel("score")
plt.title("Toy 6 · regularization changes policy choice")
plt.legend()
plt.show()

▶ What you'll see: raw Q likes the shifted policy, but the regularized objective chooses the middle policy.


## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for tabular offline RL arrays, probabilities, and deterministic checks.
import matplotlib.pyplot as plt # load Matplotlib for small visual diagnostics.
np.random.seed(0) # make random examples reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Count logged support

**Goal.** Build the smallest offline dataset and count how often each action appears, because support determines where a value estimate is grounded. We build it in 2 steps.

In [ ]:
states_b1 = np.array([0, 0, 0, 0, 1, 1, 1, 1]) # logged states from the behavior policy.
actions_b1 = np.array([0, 0, 0, 1, 0, 1, 1, 1]) # logged actions; each row is fixed historical data.
counts_b1 = np.zeros((2, 2), dtype=int) # allocate a state-action count table.
for s_b1, a_b1 in zip(states_b1, actions_b1):
    counts_b1[s_b1, a_b1] += 1 # increment the observed state-action cell.

print("counts:\n", counts_b1)

assert counts_b1.tolist() == [[3, 1], [1, 3]]

▶ What you'll see: each state has one common action and one rare action.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact support heatmap.
plt.imshow(counts_b1, cmap="Blues", aspect="auto") # visualize counts as color.
plt.colorbar(label="logged count") # add a count scale.
plt.xticks([0, 1], ["a0", "a1"]); plt.yticks([0, 1], ["s0", "s1"]) # label state-action axes.
plt.title("Basic 1: logged support") # title the plot.
plt.show() # display the heatmap.

▶ What you'll see: dark cells are better supported by the fixed dataset.

👀 Takeaway: offline RL should know the behavior-policy support before trusting any improvement.

### Basic 2 — Estimate the behavior policy

**Goal.** Convert counts into μ(a|s), because the logged policy is the reference distribution for distribution-shift checks. We build it in 2 steps.

In [ ]:
counts_b2 = np.array([[3, 1], [1, 3]], dtype=float) # reuse the support table from Basic 1.
mu_b2 = counts_b2 / counts_b2.sum(axis=1, keepdims=True) # normalize each state row to probabilities.

print("behavior policy μ:\n", np.round(mu_b2, 3)) # inspect empirical action probabilities.

assert np.allclose(mu_b2[0], [0.75, 0.25])

▶ What you'll see: state 0 chooses action 0 with probability 0.75 in the log.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact probability plot.
plt.bar(["μ(a0|s0)", "μ(a1|s0)"], mu_b2[0], color=["seagreen", "gray"]) # show one state's behavior policy.
plt.ylim(0, 1); plt.ylabel("probability") # keep probability scale readable.
plt.title("Basic 2: behavior policy in state 0") # title the figure.
plt.show() # display the bars.

▶ What you'll see: the logged policy strongly favors action 0 in state 0.

👀 Takeaway: μ is the data-generating policy that later regularization tries not to leave too aggressively.

### Basic 3 — Compute a discounted return

**Goal.** Add delayed rewards with discounting, because RL evaluates consequence rather than only immediate reward. We build it in 2 steps.

In [ ]:
rewards_b3 = np.array([1.0, 0.0, 2.0]) # define a three-step reward path.
gamma_b3 = 0.9 # choose the discount factor.
discounts_b3 = gamma_b3 ** np.arange(len(rewards_b3)) # compute 1, γ, γ².

print("discounts:", np.round(discounts_b3, 3)) # inspect future weights.

▶ What you'll see: later rewards receive smaller weights.

In [ ]:
G_b3 = float(np.sum(discounts_b3 * rewards_b3)) # compute the discounted return.

print("return G:", round(G_b3, 3)) # inspect the consequence total.

assert round(G_b3, 3) == 2.620
plt.figure(figsize=(4, 3)) # create a contribution plot.
plt.bar(["t0", "t1", "t2"], discounts_b3 * rewards_b3, color="teal") # show discounted reward pieces.
plt.title("Basic 3: discounted return pieces") # title the plot.
plt.ylabel("discounted reward") # label the contribution axis.
plt.show() # display the plot.

▶ What you'll see: the delayed reward still matters, but it is worth 1.62 instead of 2.0.

👀 Takeaway: return is a discounted sum of consequences, not a synonym for immediate reward.

### Basic 4 — Build one Bellman target

**Goal.** Compute `r + γV(s')`, because bootstrapping is how value learning uses a one-step transition to estimate future consequence. We build it in 2 steps.

In [ ]:
r_b4 = 1.0 # observed immediate reward.
gamma_b4 = 0.9 # discount factor.
V_next_b4 = 0.8 # current next-state value estimate.
target_b4 = r_b4 + gamma_b4 * V_next_b4 # one-step Bellman target.

print("target:", round(target_b4, 3)) # inspect the target.

assert round(target_b4, 3) == 1.720

▶ What you'll see: reward 1 plus discounted future value 0.72 gives target 1.72.

In [ ]:
plt.figure(figsize=(4, 3)) # create a bar chart for the target pieces.
plt.bar(["r", "γV(s')", "target"], [r_b4, gamma_b4 * V_next_b4, target_b4], color=["gray", "orange", "seagreen"]) # compare pieces.
plt.title("Basic 4: one-step target") # title the figure.
plt.ylabel("value") # label the scale.
plt.show() # display the bars.

▶ What you'll see: the target is the sum of immediate and discounted future terms.

👀 Takeaway: a Bellman target is a compact estimate of delayed consequence.

### Basic 5 — Update one Q entry

**Goal.** Move an old Q value partway toward a target, because incremental updates avoid letting one transition overwrite the whole table. We build it in 2 steps.

In [ ]:
q_old_b5 = 0.4 # old estimate before seeing the target.
target_b5 = 1.72 # Bellman target from Basic 4.
alpha_b5 = 0.5 # learning rate.
td_error_b5 = target_b5 - q_old_b5 # temporal-difference error.

print("TD error:", round(td_error_b5, 3)) # inspect how far the target is from the old estimate.

▶ What you'll see: the target is 1.32 above the current Q value.

In [ ]:
q_new_b5 = q_old_b5 + alpha_b5 * td_error_b5 # partial Bellman update.

print("new Q:", round(q_new_b5, 3)) # inspect the updated estimate.

assert round(q_new_b5, 3) == 1.060
plt.figure(figsize=(4, 3)) # create a before-target-after plot.
plt.bar(["old", "target", "new"], [q_old_b5, target_b5, q_new_b5], color=["gray", "black", "teal"]) # show movement toward target.
plt.title("Basic 5: partial Q update") # title the plot.
plt.ylabel("Q value") # label the value axis.
plt.show() # display the plot.

▶ What you'll see: the new Q lies halfway between the old value and the target.

👀 Takeaway: TD learning changes an estimate by a controlled fraction of its target error.

### Basic 6 — Softmax turns logits into a policy

**Goal.** Convert logits to action probabilities, because policy improvement often changes probabilities rather than choosing a single deterministic action immediately. We build it in 2 steps.

In [ ]:
logits_b6 = np.array([1.0, 0.0]) # unnormalized action preferences.
exp_b6 = np.exp(logits_b6 - np.max(logits_b6)) # stable exponentials for softmax.
pi_b6 = exp_b6 / exp_b6.sum() # normalize into probabilities.

print("policy:", np.round(pi_b6, 3)) # inspect action probabilities.

assert np.allclose(np.round(pi_b6, 3), [0.731, 0.269])

▶ What you'll see: a one-logit advantage becomes about 73% action probability.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact policy plot.
plt.bar(["a0", "a1"], pi_b6, color="purple") # show softmax probabilities.
plt.ylim(0, 1); plt.ylabel("π(a|s)") # use a probability axis.
plt.title("Basic 6: softmax policy") # title the plot.
plt.show() # display the plot.

▶ What you'll see: probabilities sum to one but still preserve the logit ordering.

👀 Takeaway: policy logits become action probabilities through exponentiate-and-normalize logic.

### Basic 7 — Expected reward under a policy

**Goal.** Weight action rewards by policy probabilities, because a stochastic policy's consequence is an expectation. We build it in 2 steps.

In [ ]:
pi_b7 = np.array([0.731, 0.269]) # policy probabilities from Basic 6 rounded for readability.
rewards_b7 = np.array([2.0, 0.0]) # action rewards in one state.
contrib_b7 = pi_b7 * rewards_b7 # probability-weighted reward pieces.

print("contributions:", np.round(contrib_b7, 3)) # inspect each action's contribution.

▶ What you'll see: action 0 contributes all the expected reward because action 1 has reward zero.

In [ ]:
expected_b7 = float(np.sum(contrib_b7)) # expected reward under the stochastic policy.

print("expected reward:", round(expected_b7, 3)) # inspect the expectation.

assert round(expected_b7, 3) == 1.462
plt.figure(figsize=(4, 3)) # create a contribution chart.
plt.bar(["a0", "a1"], contrib_b7, color="seagreen") # show weighted reward by action.
plt.title("Basic 7: expected reward contributions") # title the figure.
plt.ylabel("π(a)R(a)") # label the contribution axis.
plt.show() # display the chart.

▶ What you'll see: expected reward is the sum of probability-weighted action outcomes.

👀 Takeaway: changing policy probability mass changes expected consequence.

### Basic 8 — Measure policy shift with ratios

**Goal.** Compare π to μ with π/μ, because large ratios identify actions the learned policy uses more than the dataset did. We build it in 2 steps.

In [ ]:
mu_b8 = np.array([0.75, 0.25]) # behavior policy in state 0.
pi_b8 = np.array([0.10, 0.90]) # shifted learned policy in state 0.
ratios_b8 = pi_b8 / mu_b8 # importance-style shift ratios.

print("ratios π/μ:", np.round(ratios_b8, 2)) # inspect distribution shift by action.

assert round(float(ratios_b8[1]), 2) == 3.60

▶ What you'll see: action 1 is used 3.6 times more by π than by μ.

In [ ]:
plt.figure(figsize=(4, 3)) # create a ratio plot.
plt.bar(["a0", "a1"], ratios_b8, color=["gray", "crimson"]) # visualize shift by action.
plt.axhline(1, color="black", linestyle="--") # mark no-shift reference.
plt.title("Basic 8: policy shift ratios") # title the plot.
plt.ylabel("π/μ") # label the ratio axis.
plt.show() # display the plot.

▶ What you'll see: ratios above 1 indicate actions amplified relative to the log.

👀 Takeaway: offline risk rises when π puts much more mass than μ on weakly sampled actions.

### Basic 9 — Add a support penalty to Q

**Goal.** Lower values for rare actions, because conservative offline value learning should not let unsupported estimates dominate. We build it in 2 steps.

In [ ]:
Q_b9 = np.array([1.0, 3.0]) # raw action values in state 0.
mu_b9 = np.array([0.75, 0.25]) # behavior support in state 0.
penalty_b9 = 0.8 * (1.0 - mu_b9) # rare actions get larger penalties.

print("penalty:", np.round(penalty_b9, 3)) # inspect support-aware penalty sizes.

▶ What you'll see: action 1 receives the larger penalty because it has lower behavior support.

In [ ]:
Q_cons_b9 = Q_b9 - penalty_b9 # conservative values after subtracting support penalty.

print("conservative Q:", np.round(Q_cons_b9, 3)) # inspect adjusted values.

assert np.allclose(np.round(Q_cons_b9, 3), [0.800, 2.400])
plt.figure(figsize=(4, 3)) # create a raw-vs-conservative chart.
plt.bar(["raw a0", "raw a1", "cons a0", "cons a1"], [Q_b9[0], Q_b9[1], Q_cons_b9[0], Q_cons_b9[1]], color=["gray", "gray", "teal", "crimson"]) # compare values.
plt.xticks(rotation=20); plt.ylabel("Q") # label the value scale.
plt.title("Basic 9: conservative adjustment") # title the figure.
plt.show() # display the chart.

▶ What you'll see: the rare action's optimism is reduced more strongly.

👀 Takeaway: conservatism is a value-side defense against extrapolating from weak support.

### Basic 10 — Score a behavior-regularized policy

**Goal.** Subtract a KL distance from expected Q, because offline policy improvement should pay a cost for moving away from the data. We build it in 2 steps.

In [ ]:
mu_b10 = np.array([0.75, 0.25]) # behavior policy reference.
pi_b10 = np.array([0.50, 0.50]) # candidate improved policy.
Q_b10 = np.array([0.8, 2.4]) # conservative action values.
kl_b10 = float(np.sum(pi_b10 * np.log(pi_b10 / mu_b10))) # KL(π||μ).
value_b10 = float(pi_b10 @ Q_b10) # expected conservative value.

print("value:", round(value_b10, 3), "KL:", round(kl_b10, 3)) # inspect both objective pieces.

▶ What you'll see: the policy gains value but is measurably away from μ.

In [ ]:
beta_b10 = 1.2 # behavior-regularization strength.
objective_b10 = value_b10 - beta_b10 * kl_b10 # regularized policy score.

print("regularized score:", round(objective_b10, 3)) # inspect Q minus distance cost.

assert round(objective_b10, 3) == 1.427
plt.figure(figsize=(4, 3)) # create an objective breakdown chart.
plt.bar(["Eπ[Q]", "β KL", "score"], [value_b10, beta_b10 * kl_b10, objective_b10], color=["teal", "crimson", "seagreen"]) # compare pieces.
plt.title("Basic 10: behavior-regularized score") # title the plot.
plt.ylabel("objective units") # label the scale.
plt.show() # display the chart.

▶ What you'll see: the KL cost lowers the raw expected Q score.

👀 Takeaway: behavior regularization makes improvement a constrained optimization problem, not pure greed.

## 🟡 Easy

### Easy 1 — Tabular offline Q evaluation

**Goal.** Evaluate a target policy from fixed transitions with Bellman-style sweeps, because offline RL must reuse logged next states instead of sampling new ones. We build it in 3 steps.

In [ ]:
states_e1 = np.array([0, 0, 0, 0, 1, 1, 1, 1]) # fixed logged states.
actions_e1 = np.array([0, 0, 0, 1, 0, 1, 1, 1]) # fixed logged actions.
rewards_e1 = np.array([1.0, 1.2, 0.8, 1.5, 0.2, 2.0, 1.8, 2.2]) # fixed rewards.
next_states_e1 = np.array([0, 0, 1, 1, 0, 1, 1, 1]) # fixed next states.
pi_e1 = np.array([[0.75, 0.25], [0.20, 0.80]]) # target policy to evaluate.
Q_e1 = np.zeros((2, 2)) # initialize tabular action values.

print("transitions:", len(states_e1)) # inspect dataset size.

▶ What you'll see: evaluation starts from eight fixed transitions and no online exploration.

In [ ]:
gamma_e1 = 0.9 # discount for future consequence.
alpha_e1 = 0.25 # small learning rate for stable repeated sweeps.
for sweep_e1 in range(80): # repeatedly replay the fixed dataset.
    for s_e1, a_e1, r_e1, ns_e1 in zip(states_e1, actions_e1, rewards_e1, next_states_e1):
        v_next_e1 = float(pi_e1[ns_e1] @ Q_e1[ns_e1]) # target policy value at the logged next state.
        target_e1 = r_e1 + gamma_e1 * v_next_e1 # expected-SARSA target.
        Q_e1[s_e1, a_e1] += alpha_e1 * (target_e1 - Q_e1[s_e1, a_e1]) # update only logged action.

print("evaluated Q:\n", np.round(Q_e1, 3)) # inspect learned values.

assert Q_e1[1, 1] > Q_e1[1, 0]

▶ What you'll see: the evaluated Q table grows only from replayed logged transitions, with state 1 action 1 highest.

In [ ]:
plt.figure(figsize=(4, 3)) # create a Q heatmap.
plt.imshow(Q_e1, cmap="viridis", aspect="auto") # visualize learned action values.
plt.colorbar(label="Qπ(s,a)") # add value scale.
plt.xticks([0, 1], ["a0", "a1"]); plt.yticks([0, 1], ["s0", "s1"]) # label axes.
plt.title("Easy 1: offline policy evaluation") # title the plot.
plt.show() # display the heatmap.

▶ What you'll see: the high-reward action in state 1 gets the larger evaluated value.

👀 Takeaway: offline evaluation backs up through logged transitions and cannot update actions absent from the log.

### Easy 2 — Show extrapolation error from a greedy max

**Goal.** Compare a max backup to a behavior-weighted backup, because max can select unsupported overestimates in offline data. We build it in 3 steps.

In [ ]:
Q_next_e2 = np.array([1.0, 5.0]) # next-state values where action 1 is an unsupported overestimate.
mu_next_e2 = np.array([0.95, 0.05]) # behavior policy almost never tried action 1.
r_e2 = 0.5 # current reward.
gamma_e2 = 0.9 # discount factor.

print("next Q:", Q_next_e2, "behavior μ:", mu_next_e2) # inspect support and values.

▶ What you'll see: the largest value belongs to the action with almost no support.

In [ ]:
max_target_e2 = r_e2 + gamma_e2 * np.max(Q_next_e2) # standard greedy target.
behavior_target_e2 = r_e2 + gamma_e2 * float(mu_next_e2 @ Q_next_e2) # support-weighted target.

print("max target:", round(max_target_e2, 3)) # inspect greedy backup.
print("behavior-weighted target:", round(behavior_target_e2, 3)) # inspect support-respecting backup.

assert round(max_target_e2, 3) == 5.000

▶ What you'll see: the max target is much larger than the behavior-weighted target because it trusts the rare high value.

In [ ]:
plt.figure(figsize=(4, 3)) # create a backup comparison.
plt.bar(["max backup", "μ-weighted backup"], [max_target_e2, behavior_target_e2], color=["crimson", "teal"]) # compare targets.
plt.ylabel("target value") # label target scale.
plt.title("Easy 2: unsupported max overestimation") # title the figure.
plt.show() # display the bars.

▶ What you'll see: the greedy max target is much higher because it trusts the rare action's value.

👀 Takeaway: offline backups need support control because a max over noisy Q estimates selects unsupported optimism.

### Easy 3 — Conservative Q iteration on a fixed dataset

**Goal.** Train Q values while subtracting a support penalty, because conservative updates reduce the incentive to exploit rare actions. We build it in 3 steps.

In [ ]:
states_e3 = np.array([0, 0, 0, 0, 1, 1, 1, 1]) # fixed logged states.
actions_e3 = np.array([0, 0, 0, 1, 0, 1, 1, 1]) # fixed logged actions.
rewards_e3 = np.array([1.0, 1.2, 0.8, 1.5, 0.2, 2.0, 1.8, 2.2]) # fixed rewards.
next_states_e3 = np.array([0, 0, 1, 1, 0, 1, 1, 1]) # fixed next states.
counts_e3 = np.array([[3, 1], [1, 3]], dtype=float) # behavior counts.
mu_e3 = counts_e3 / counts_e3.sum(axis=1, keepdims=True) # behavior probabilities.
Q_e3 = np.zeros((2, 2)) # initialize Q table.

print("μ:\n", mu_e3) # inspect support reference.

▶ What you'll see: support is uneven across actions in both states.

In [ ]:
penalty_e3 = 0.7 * (1 - mu_e3) # conservative penalty larger for rare actions.
for sweep_e3 in range(90): # replay fixed transitions.
    for s_e3, a_e3, r_e3, ns_e3 in zip(states_e3, actions_e3, rewards_e3, next_states_e3):
        target_e3 = r_e3 + 0.9 * np.max(Q_e3[ns_e3] - penalty_e3[ns_e3]) # conservative next-action max.
        Q_e3[s_e3, a_e3] += 0.2 * (target_e3 - Q_e3[s_e3, a_e3]) # update logged state-action.
Q_safe_e3 = Q_e3 - penalty_e3 # final conservative scores used for action choice.

print("raw Q:\n", np.round(Q_e3, 3)) # inspect learned raw values.
print("conservative scores:\n", np.round(Q_safe_e3, 3)) # inspect penalized values.

▶ What you'll see: raw Q values are high, but conservative scores subtract more from low-support actions.

In [ ]:
plt.figure(figsize=(4, 3)) # create a conservative-score heatmap.
plt.imshow(Q_safe_e3, cmap="viridis", aspect="auto") # visualize values after penalty.
plt.colorbar(label="Q - penalty") # add conservative value scale.
plt.xticks([0, 1], ["a0", "a1"]); plt.yticks([0, 1], ["s0", "s1"]) # label axes.
plt.title("Easy 3: conservative Q scores") # title the plot.
plt.show() # display the heatmap.

▶ What you'll see: rare actions are not allowed to win solely because of optimistic bootstrapping.

👀 Takeaway: conservative Q learning modifies the backup so high value must survive a support penalty.

### Easy 4 — Choose among behavior-regularized candidates

**Goal.** Select a policy by maximizing `Eπ[Q] - β KL(π||μ)`, because offline improvement is safer when it remains near logged behavior. We build it in 3 steps.

In [ ]:
mu_e4 = np.array([0.75, 0.25]) # behavior policy in one state.
Q_e4 = np.array([0.8, 2.4]) # conservative action values.
candidates_e4 = np.array([[0.75, 0.25], [0.50, 0.50], [0.20, 0.80]]) # candidate target policies.
beta_e4 = 1.2 # distance penalty strength.

print("candidates:\n", candidates_e4) # inspect candidate policies.

▶ What you'll see: candidates range from behavior-matching to strongly shifted.

In [ ]:
values_e4 = candidates_e4 @ Q_e4 # expected Q for each candidate.
kl_e4 = np.sum(candidates_e4 * np.log(np.maximum(candidates_e4, 1e-9) / mu_e4), axis=1) # KL distance from μ.
scores_e4 = values_e4 - beta_e4 * kl_e4 # regularized objective.
best_e4 = int(np.argmax(scores_e4)) # choose the best candidate.

print("values:", np.round(values_e4, 3)) # inspect raw value.
print("scores:", np.round(scores_e4, 3), "best:", best_e4) # inspect regularized winner.

assert best_e4 == 1

▶ What you'll see: the printed scores show the middle candidate winning after the KL cost is applied.

In [ ]:
plt.figure(figsize=(5, 3)) # create a candidate score plot.
plt.bar(["near μ", "middle", "shifted"], scores_e4, color=["gray", "teal", "crimson"]) # compare regularized objectives.
plt.title("Easy 4: behavior-regularized selection") # title the plot.
plt.ylabel("Eπ[Q] - β KL") # label objective axis.
plt.show() # display the plot.

▶ What you'll see: the middle policy wins after paying for distance from behavior.

👀 Takeaway: behavior regularization can prefer moderate improvement over unsupported greed.

### Easy 5 — Estimate a policy value with importance weights

**Goal.** Reweight logged rewards by π/μ, because off-policy evaluation estimates how target-policy outcomes differ from behavior-policy data. We build it in 3 steps.

In [ ]:
rewards_e5 = np.array([1.0, 1.2, 0.8, 1.5]) # one-state logged rewards.
actions_e5 = np.array([0, 0, 0, 1]) # behavior actions in that state.
mu_e5 = np.array([0.75, 0.25]) # behavior probabilities.
pi_e5 = np.array([0.50, 0.50]) # target policy probabilities.
weights_e5 = pi_e5[actions_e5] / mu_e5[actions_e5] # importance weights for each logged row.

print("weights:", np.round(weights_e5, 3)) # inspect how rows are reweighted.

▶ What you'll see: rare action-1 data receives a larger weight under the target policy.

In [ ]:
ordinary_e5 = float(np.mean(weights_e5 * rewards_e5)) # ordinary importance-sampling estimate.
weighted_e5 = float(np.sum(weights_e5 * rewards_e5) / np.sum(weights_e5)) # self-normalized estimate.

print("ordinary IS:", round(ordinary_e5, 3), "weighted IS:", round(weighted_e5, 3)) # compare estimators.

assert round(weighted_e5, 3) == 1.250

▶ What you'll see: ordinary and self-normalized estimates agree on this balanced toy log.

In [ ]:
plt.figure(figsize=(4, 3)) # create an estimator comparison chart.
plt.bar(["behavior mean", "ordinary IS", "weighted IS"], [np.mean(rewards_e5), ordinary_e5, weighted_e5], color=["gray", "orange", "teal"]) # compare value estimates.
plt.title("Easy 5: off-policy value estimates") # title the plot.
plt.ylabel("estimated one-step value") # label the value scale.
plt.show() # display the chart.

▶ What you'll see: changing from μ to π changes which logged rows count most.

👀 Takeaway: importance weighting corrects distribution shift in expectation but can become high variance when ratios are large.

## 🔴 Advanced

### Advanced 1 — Sweep conservatism strength

**Goal.** Vary the support-penalty coefficient and watch action choice change, because conservatism is a bias-safety knob. We build it in 4 steps.

In [ ]:
Q_raw_a1 = np.array([[1.0, 3.0], [0.5, 2.0]]) # optimistic raw Q table.
mu_a1 = np.array([[0.75, 0.25], [0.25, 0.75]]) # behavior support table.
lambdas_a1 = np.array([0.0, 1.0, 3.0, 5.0]) # conservatism strengths.

print("lambda grid:", lambdas_a1) # inspect penalty strengths.

▶ What you'll see: λ=0 is unconservative and larger λ penalizes weak support more.

In [ ]:
chosen_a1 = [] # store greedy action in state 0 for each λ.
margin_a1 = [] # store conservative a1-a0 margin.
for lam_a1 in lambdas_a1:
    Q_cons_a1 = Q_raw_a1 - lam_a1 * (1 - mu_a1) # support-aware conservative score.
    chosen_a1.append(int(np.argmax(Q_cons_a1[0]))) # greedy action after penalty.
    margin_a1.append(float(Q_cons_a1[0, 1] - Q_cons_a1[0, 0])) # advantage of rare action.

print("chosen actions in s0:", chosen_a1) # inspect policy shift as λ changes.
print("a1-a0 margins:", np.round(margin_a1, 3)) # inspect when rare action stops winning.

assert chosen_a1[-1] == 0

▶ What you'll see: the rare action wins at low λ, but the margin shrinks and flips at high λ.

In [ ]:
plt.figure(figsize=(5, 3)) # create a margin sweep plot.
plt.plot(lambdas_a1, margin_a1, marker="o", color="crimson") # plot rare-action margin by λ.
plt.axhline(0, color="black", linestyle="--") # mark indifference.
plt.title("Advanced 1: conservatism flips unsupported greed") # title the plot.
plt.xlabel("conservatism λ") # label penalty strength.
plt.ylabel("Q_cons(s0,a1) - Q_cons(s0,a0)") # label margin.
plt.show() # display the curve.

▶ What you'll see: as λ grows, the rare action's advantage shrinks and eventually becomes negative.

In [ ]:
final_scores_a1 = Q_raw_a1[0] - lambdas_a1[-1] * (1 - mu_a1[0]) # final conservative scores for state 0.

print("final state-0 scores:", np.round(final_scores_a1, 3)) # inspect the actual values that flipped the action.

assert np.argmax(final_scores_a1) == 0

▶ What you'll see: the final high-λ scores make action 0 larger than action 1 in state 0.



👀 Takeaway: stronger conservatism increases pessimism for weak-support actions and can deliberately sacrifice apparent value for robustness.

### Advanced 2 — Sweep behavior regularization β

**Goal.** Vary β in `Eπ[Q] - β KL`, because β controls how far improvement is allowed to move from logged behavior. We build it in 4 steps.

In [ ]:
mu_a2 = np.array([0.75, 0.25]) # behavior policy reference.
Q_a2 = np.array([0.8, 2.4]) # conservative values.
candidates_a2 = np.array([[0.75, 0.25], [0.50, 0.50], [0.20, 0.80]]) # policy candidates.
betas_a2 = np.array([0.0, 0.5, 1.2, 3.0]) # regularization strengths.

print("beta grid:", betas_a2) # inspect sweep settings.

▶ What you'll see: β=0 ignores behavior distance; larger β makes distance expensive.

In [ ]:
values_a2 = candidates_a2 @ Q_a2 # expected conservative Q.
kl_a2 = np.sum(candidates_a2 * np.log(np.maximum(candidates_a2, 1e-9) / mu_a2), axis=1) # candidate KL distances.
winners_a2 = [] # store winner index for each beta.
for beta_a2 in betas_a2:
    score_a2 = values_a2 - beta_a2 * kl_a2 # regularized objective at this beta.
    winners_a2.append(int(np.argmax(score_a2))) # choose best candidate.

print("KL:", np.round(kl_a2, 3)) # inspect candidate distances.
print("winners:", winners_a2) # inspect how regularization changes selected policy.

assert winners_a2[0] == 2 and winners_a2[-1] == 0

▶ What you'll see: the winner moves from shifted to middle to behavior-like as β increases.

In [ ]:
shift_prob_a2 = candidates_a2[winners_a2, 1] # probability of rare/high-value action chosen by each beta.
plt.figure(figsize=(5, 3)) # create beta sweep plot.
plt.plot(betas_a2, shift_prob_a2, marker="o", color="purple") # plot selected action-1 probability.
plt.title("Advanced 2: β controls policy shift") # title the figure.
plt.xlabel("β") # label regularization strength.
plt.ylabel("chosen π(a1|s)") # label selected shift.
plt.ylim(0, 1) # probability bounds.
plt.show() # display the plot.

▶ What you'll see: high β pulls the chosen policy back toward behavior.

In [ ]:
print("chosen policies:\n", candidates_a2[winners_a2]) # inspect the actual selected policies by beta.

▶ What you'll see: the chosen policies become progressively closer to μ as regularization strengthens.



👀 Takeaway: β is the policy-side safety knob: low β is greedy, high β is behavior-cloning-like.

### Advanced 3 — Detect unsupported target actions

**Goal.** Flag state-action pairs where π is large but μ is tiny, because those are the places offline evaluation and improvement are most fragile. We build it in 4 steps.

In [ ]:
mu_a3 = np.array([[0.90, 0.10], [0.05, 0.95], [0.50, 0.50]]) # behavior support over three states.
pi_a3 = np.array([[0.20, 0.80], [0.60, 0.40], [0.40, 0.60]]) # target policy probabilities.
ratio_a3 = pi_a3 / np.maximum(mu_a3, 1e-9) # distribution-shift ratios.

print("ratios:\n", np.round(ratio_a3, 2)) # inspect shift table.

▶ What you'll see: state 1 action 0 has especially large target-over-behavior ratio.

In [ ]:
risk_mask_a3 = (pi_a3 > 0.30) & (mu_a3 < 0.15) # simple support-risk rule.
risky_pairs_a3 = np.argwhere(risk_mask_a3) # list risky state-action coordinates.

print("risky pairs [state, action]:", risky_pairs_a3.tolist()) # inspect flagged pairs.

assert risky_pairs_a3.tolist() == [[0, 1], [1, 0]]

▶ What you'll see: two state-action pairs are flagged because π is large where μ is small.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create a ratio heatmap.
plt.imshow(ratio_a3, cmap="Reds", aspect="auto") # visualize large ratios as darker red.
plt.colorbar(label="π/μ") # add ratio scale.
plt.xticks([0, 1], ["a0", "a1"]); plt.yticks([0, 1, 2], ["s0", "s1", "s2"]) # label axes.
plt.title("Advanced 3: unsupported target-action ratios") # title the plot.
plt.show() # display the heatmap.

▶ What you'll see: the flagged risky actions correspond to the darkest high-ratio cells.

In [ ]:
risk_count_a3 = int(risk_mask_a3.sum()) # count how many unsupported choices the target policy makes.

print("number of risky state-action pairs:", risk_count_a3) # inspect summary risk.

assert risk_count_a3 == 2

▶ What you'll see: the risk count summarizes how many unsupported target choices need caution.



👀 Takeaway: support diagnostics should be run before trusting an offline policy's apparent value.

### Advanced 4 — Compare ordinary and clipped importance sampling

**Goal.** Clip large importance weights and compare estimates, because clipping trades bias for lower variance under severe distribution shift. We build it in 4 steps.

In [ ]:
rewards_a4 = np.array([1.0, 0.8, 1.2, 3.0, 2.8]) # logged one-step rewards.
mu_probs_a4 = np.array([0.9, 0.9, 0.9, 0.1, 0.1]) # behavior probabilities of logged actions.
pi_probs_a4 = np.array([0.4, 0.4, 0.4, 0.6, 0.6]) # target probabilities of those same logged actions.
weights_a4 = pi_probs_a4 / mu_probs_a4 # importance weights.

print("weights:", np.round(weights_a4, 2)) # inspect high-variance ratios.

▶ What you'll see: the rare behavior actions receive weights of 6.

In [ ]:
ordinary_a4 = float(np.mean(weights_a4 * rewards_a4)) # ordinary IS estimate.
clipped_weights_a4 = np.minimum(weights_a4, 2.0) # clip extreme ratios.
clipped_a4 = float(np.mean(clipped_weights_a4 * rewards_a4)) # clipped estimate.

print("ordinary:", round(ordinary_a4, 3), "clipped:", round(clipped_a4, 3)) # compare estimates.

assert ordinary_a4 > clipped_a4

▶ What you'll see: clipping lowers the estimate by capping the largest importance weights.

In [ ]:
plt.figure(figsize=(5, 3)) # create a weight comparison plot.
plt.bar(np.arange(len(weights_a4)) - 0.18, weights_a4, width=0.36, label="raw", color="crimson") # raw IS weights.
plt.bar(np.arange(len(weights_a4)) + 0.18, clipped_weights_a4, width=0.36, label="clipped", color="teal") # clipped weights.
plt.title("Advanced 4: clipping large importance weights") # title the plot.
plt.xlabel("logged row") # label row axis.
plt.ylabel("weight") # label weight scale.
plt.legend() # show labels.
plt.show() # display grouped bars.

▶ What you'll see: clipping caps the two extreme rows that would dominate the estimate.

In [ ]:
variance_proxy_a4 = float(np.var(weights_a4 * rewards_a4)) # simple variability proxy before clipping.
clipped_variance_proxy_a4 = float(np.var(clipped_weights_a4 * rewards_a4)) # variability proxy after clipping.

print("variance proxy raw/clipped:", round(variance_proxy_a4, 3), round(clipped_variance_proxy_a4, 3)) # inspect variance reduction.

assert clipped_variance_proxy_a4 < variance_proxy_a4

▶ What you'll see: the clipped weighted rewards have much lower variance proxy than the raw weighted rewards.



👀 Takeaway: clipping can stabilize off-policy estimates, but the lower-variance answer is biased by design.

### Advanced 5 — Train conservative and behavior-regularized policies together

**Goal.** Combine conservative scores with KL-regularized policy selection, because practical offline RL often uses both value pessimism and policy constraints. We build it in 5 steps.

In [ ]:
Q_raw_a5 = np.array([[1.0, 3.0], [0.5, 2.0]]) # raw learned Q table.
mu_a5 = np.array([[0.75, 0.25], [0.25, 0.75]]) # empirical behavior policy.
lambda_a5 = 1.0 # conservative value penalty strength.
beta_a5 = 1.2 # behavior-regularization strength.
Q_cons_a5 = Q_raw_a5 - lambda_a5 * (1 - mu_a5) # conservative values.

print("conservative Q:\n", np.round(Q_cons_a5, 3)) # inspect value pessimism.

▶ What you'll see: weak-support actions are lowered before policy selection.

In [ ]:
candidate_grid_a5 = np.array([[0.80, 0.20], [0.60, 0.40], [0.40, 0.60], [0.20, 0.80]]) # candidate policies per state.
chosen_pi_a5 = np.zeros_like(mu_a5) # allocate selected policy.
chosen_scores_a5 = np.zeros(2) # store selected objective score by state.
for s_a5 in range(2):
    vals_a5 = candidate_grid_a5 @ Q_cons_a5[s_a5] # expected conservative Q for candidates.
    kls_a5 = np.sum(candidate_grid_a5 * np.log(np.maximum(candidate_grid_a5, 1e-9) / mu_a5[s_a5]), axis=1) # KL from behavior.
    scores_a5 = vals_a5 - beta_a5 * kls_a5 # regularized scores.
    idx_a5 = int(np.argmax(scores_a5)) # choose best candidate for this state.
    chosen_pi_a5[s_a5] = candidate_grid_a5[idx_a5] # store selected policy.
    chosen_scores_a5[s_a5] = scores_a5[idx_a5] # store selected score.

print("chosen policy:\n", chosen_pi_a5) # inspect the final policy.

▶ What you'll see: each state chooses the candidate with the best conservative value minus KL cost.

In [ ]:
shift_a5 = np.sum(np.abs(chosen_pi_a5 - mu_a5), axis=1) # L1 shift from behavior by state.

print("L1 shift by state:", np.round(shift_a5, 3)) # inspect how far the selected policy moved.

assert np.all(shift_a5 <= 0.7)

▶ What you'll see: the L1 shifts confirm the learned policy moved, but not arbitrarily far from behavior.

In [ ]:
plt.figure(figsize=(5, 3)) # create final policy comparison for state 0.
x_a5 = np.arange(2) # action positions.
plt.bar(x_a5 - 0.18, mu_a5[0], width=0.36, label="behavior μ", color="gray") # behavior bars.
plt.bar(x_a5 + 0.18, chosen_pi_a5[0], width=0.36, label="offline π", color="teal") # learned policy bars.
plt.xticks(x_a5, ["a0", "a1"]); plt.ylim(0, 1) # label actions and probability scale.
plt.title("Advanced 5: conservative + behavior-regularized π") # title the plot.
plt.legend(); plt.show() # display comparison.

▶ What you'll see: the learned policy shifts toward better actions but does not collapse onto the rare action.

In [ ]:
score_sum_a5 = float(np.sum(chosen_scores_a5)) # summarize selected regularized objectives.

print("sum selected scores:", round(score_sum_a5, 3)) # inspect numeric objective summary.

assert score_sum_a5 > 1.0

▶ What you'll see: the summed score is positive, showing the selected policies kept useful regularized value.



👀 Takeaway: robust offline RL usually needs both pessimistic values and a policy-distance constraint to manage distribution shift.